# COVID-19 Prediction Model Using Machine Learning

In [2]:
# import important libaries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler , OneHotEncoder , LabelEncoder

In [3]:
# read data from local disk
df = pd.read_csv(r"C:\Users\Lenovo\Downloads\covid_toy.csv")
df

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No
...,...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore,No
96,51,Female,101.0,Strong,Kolkata,Yes
97,20,Female,101.0,Mild,Bangalore,No
98,5,Female,98.0,Strong,Mumbai,No


In [4]:
# Handel the missing data
df["fever"].fillna(df["fever"].mean() , inplace = True)

l = LabelEncoder()
df["has_covid"] = l.fit_transform(df["has_covid"])
df.dropna(inplace=True)
df.isnull().sum()



C:\Users\Lenovo\AppData\Local\Temp\ipykernel_15676\1380376076.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["fever"].fillna(df["fever"].mean() , inplace = True)


age          0
gender       0
fever        0
cough        0
city         0
has_covid    0
dtype: int64

In [5]:
# Split target and feature columns
X = df.drop(columns = ["has_covid"] , axis =1)
y = df[["has_covid"]]

In [6]:
# Initialize the encoder
le = LabelEncoder()
y = le.fit_transform(y)

C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [7]:
# Classify data into Numerical or Categorical
cat = X.select_dtypes(include = ["object"]).columns
num = X.select_dtypes(exclude = ["object"]).columns

In [58]:
from sklearn.compose import ColumnTransformer
pre = ColumnTransformer(
    transformers=[
        ("cat",OneHotEncoder(drop = "first"), cat),
        ("num", StandardScaler(),num)
    ]
)
pre

,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,True


In [59]:
from sklearn.compose import ColumnTransformer
pre = ColumnTransformer(
    transformers=[
        ("cat",OneHotEncoder(), cat),
        ("num", StandardScaler(),num)
    ]
)
pre

,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,True


In [60]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [61]:
# make pipeline for model and make a model

model = Pipeline(steps=[
    ("Preprocesser", pre),
    ("Logistic", LogisticRegression(
        C=0.1,          # smaller = stronger regularization
        penalty='l2',   # default but good
        solver='liblinear',  # stable for small datasets
        max_iter=1000
    ))
])

In [62]:
# Split into Training and Testing
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X,y , test_size=0.2 , random_state=42)

In [70]:
# Transform your target variables
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [83]:

model.fit(X_train , y_train)

,steps,"[('Preprocesser', ...), ('Logistic', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [84]:
# check accuarcy of model on test data
from sklearn.metrics import accuracy_score , confusion_matrix

y_pred = model.predict(X_test)
acc = accuracy_score(y_pred , y_test)
print("Accuracy of model  test : " , acc)



Accuracy of model  test :  0.75


In [85]:
# check accuarcy of model on train data

y_pred2 = model.predict(X_train)

acc2 = accuracy_score(y_pred2 , y_train)
# con = confusion_matrix(y_pred , y_train)

print("Accuracy of model  train : " , acc2)
# print("Confusion Metrix : " , con)

Accuracy of model  train :  0.6125


In [86]:
df.head(10)

,age,gender,fever,cough,city,has_covid
0,60,Male,103.000000,Mild,Kolkata,0
1,27,Male,100.000000,Mild,Delhi,1
2,42,Male,101.000000,Mild,Delhi,0
3,31,Female,98.000000,Mild,Kolkata,0
4,65,Female,101.000000,Mild,Mumbai,0
5,84,Female,100.844444,Mild,Bangalore,1
6,14,Male,101.000000,Strong,Bangalore,0
7,20,Female,100.844444,Strong,Mumbai,1
8,19,Female,100.000000,Strong,Bangalore,0
9,64,Female,101.000000,Mild,Delhi,0


In [93]:
live = pd.DataFrame({
    'age' : [14], 
    'gender' : ["Male"], 
    'fever' : [101.0], 
    'cough' : ["Strong"], 
    'city' : ["Bangalore"]
})

In [95]:
probility = model.predict_proba(live)
positive_prob = probility[0][1]

result = " has covid" if positive_prob > 0.5 else "has not covid"

print("Pridiction :" , result)
print("Positive Probility :" , positive_prob)

Pridiction : has not covid
Positive Probility : 0.4344137163045853
[0 1]


In [56]:
# Make columns 
submmision = pd.DataFrame({
    "ID "  : X_test.index,
    "Pridiction " : y_pred,
    "Actual value " : y_test
    
})

In [25]:
#  Save pridicted data in csv file
submmision.to_csv(r"C:\Users\Lenovo\Downloads\pridict_covid.csv" , index = False)